# AdventureWorks Power BI Sales Dashboard

## Project Overview

This project focuses on building a professional Power BI dashboard using the AdventureWorks sales dataset.

Unlike my previous projects, which focused more heavily on data cleaning and SQL analysis, this project is designed to practice Power BI data modeling, relationships, DAX measures, and dashboard design.

The dataset contains multiple related tables, including sales, returns, customers, products, product categories, product subcategories, territories, and calendar data. These tables will be lightly audited and cleaned in Python before being loaded into Power BI as separate base tables.

The main goal is to build an interactive dashboard that helps users understand sales performance, product performance, regional performance, customer patterns, and return behavior.

## Project Objectives

The objectives of this project are to:

- Audit the raw AdventureWorks CSV files using Python and pandas.
- Combine yearly sales files into one sales table.
- Clean missing or invalid values where appropriate.
- Preserve the separate lookup and fact tables for Power BI modeling.
- Load the cleaned tables into Power BI.
- Build relationships between fact tables and lookup tables.
- Create DAX measures for key business metrics.
- Design an interactive dashboard with slicers and cross-filtering.
- Communicate business insights through clear dashboard visuals.

## Key Skills Practiced

This project is intended to improve the following skills:

- Power BI data modeling
- Fact and dimension table relationships
- DAX measures
- Date/calendar table usage
- Interactive slicers and filters
- Dashboard layout and visual design
- Business-focused data storytelling

## Planned Workflow

1. Load raw CSV files into pandas.
2. Perform preliminary data audits.
3. Clean customer and product attribute issues.
4. Combine the yearly sales files into one sales table.
5. Export cleaned base tables.
6. Load cleaned tables into Power BI.
7. Create relationships between tables.
8. Build DAX measures.
9. Design the dashboard pages.
10. Document insights, limitations, and final deliverables.

## Project Scope

This project will focus mainly on Power BI dashboard development. Python will be used for light auditing and preparation, while Power BI will be used as the main analysis and reporting tool.

SQL will not be the main workflow in this project because SQL analysis was already covered in earlier projects. This project is focused on learning the professional Power BI workflow from cleaned base tables to an interactive dashboard.


In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine

In [ ]:
project_folder = Path.cwd()
raw_folder = project_folder / "raw_data"
clean_folder = project_folder / "clean_data"
output_folder = project_folder / "output"

In [ ]:
calendar = pd.read_csv(raw_folder / "AdventureWorks Calendar Lookup.csv")
customers = pd.read_csv(raw_folder / "AdventureWorks Customer Lookup.csv", encoding="cp1252")
product_categories = pd.read_csv(raw_folder / "AdventureWorks Product Categories Lookup.csv")
products = pd.read_csv(raw_folder / "AdventureWorks Product Lookup.csv")
product_subcategories = pd.read_csv(raw_folder / "AdventureWorks Product Subcategories Lookup.csv")
returns = pd.read_csv(raw_folder / "AdventureWorks Returns Data.csv")
sales_2020 = pd.read_csv(raw_folder / "AdventureWorks Sales Data 2020.csv")
sales_2021 = pd.read_csv(raw_folder / "AdventureWorks Sales Data 2021.csv")
sales_2022 = pd.read_csv(raw_folder / "AdventureWorks Sales Data 2022.csv")
territories = pd.read_csv(raw_folder / "AdventureWorks Territory Lookup.csv")

The customer file required `cp1252` encoding due to non-standard text characters. Text fields were reviewed for encoding issues before cleaning.


In [ ]:
combined_sales = pd.concat(
    [sales_2020, sales_2021, sales_2022],
    ignore_index=True
)


In [ ]:
def audit_df(df, name):
    print(f"Audit: {name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    display(df.head())


In [ ]:
audit_df(calendar, "calendar")
audit_df(customers, "customers")
audit_df(product_categories, "product categories")
audit_df(products, "products")
audit_df(product_subcategories, "product subcategories")
audit_df(returns, "returns")
audit_df(combined_sales, "combined sales")
audit_df(territories, "territories")


## Preliminary Data Audit Summary

The preliminary audit reviewed all raw AdventureWorks tables for row counts, missing values, duplicate rows, and sample records.

Overall, the dataset is mostly clean and suitable for Power BI modeling. The calendar, product categories, product subcategories, returns, combined sales, and territory tables contain no missing key fields and no duplicate rows.

The main data quality issues found were:

- The `customers` table contains 1 missing `CustomerKey`, which requires investigation because customer keys are used to connect customer records to the sales table.
- The `customers` table contains 2 duplicate rows, which should be reviewed before loading into the final model.
- Several customer attribute fields contain small numbers of missing values, including `Prefix`, `FirstName`, `LastName`, `BirthDate`, `MaritalStatus`, `Gender`, `EmailAddress`, `AnnualIncome`, `TotalChildren`, `EducationLevel`, `Occupation`, and `HomeOwner`.
- The `products` table contains 50 missing `ProductColor` values. Since product color is a categorical attribute, these missing values can be labeled as `Unknown` if color is used in reporting.

The yearly sales files for 2020, 2021, and 2022 were successfully combined into one sales table with 56,046 rows and no missing values or duplicate rows.

Next steps are to investigate the missing and duplicate customer records, review missing product colors, validate key relationships between fact and lookup tables, and prepare clean base tables for Power BI.


In [ ]:
customers[customers.duplicated(keep=False)]

In [ ]:
bad_customer_keys = customers.loc[customers.duplicated(keep=False), "CustomerKey"]

combined_sales[combined_sales["CustomerKey"].isin(bad_customer_keys)]


In [ ]:
customers[customers["CustomerKey"].isna()]

In [ ]:
customers_clean = customers.drop_duplicates().copy()
customers_clean = customers_clean[customers_clean["CustomerKey"].notna()].copy()
customers_clean = customers_clean.reset_index(drop=True)

Duplicate customer records were reviewed manually. The duplicated rows appeared to be invalid placeholder records, with missing customer details and a default-looking birth date of `1900-01-01`. These records were checked against the sales table before removal to ensure they were not required for sales relationship matching.


In [ ]:
customer_data_columns = [
    "Prefix",
    "FirstName",
    "LastName",
    "MaritalStatus",
    "Gender",
    "EmailAddress",
    "EducationLevel",
    "Occupation",
    "HomeOwner"
]

customers_clean[customer_data_columns] = customers_clean[customer_data_columns].fillna("Unknown")


In [ ]:
audit_df(customers_clean, "cleaned customer data")

## Customer Missing Value Handling

Missing values in the customer table were handled based on the type of field.

Categorical and text fields such as `Prefix`, `FirstName`, `LastName`, `MaritalStatus`, `Gender`, `EmailAddress`, `EducationLevel`, `Occupation`, and `HomeOwner` were filled with `Unknown`. This keeps the customer records available for filtering and grouping in Power BI while making missing categories explicit.

Numeric and date fields such as `AnnualIncome`, `TotalChildren`, and `BirthDate` were left as null values. These fields were not imputed because there was insufficient evidence to accurately estimate the missing values, and filling them with `0` or `Unknown` would distort calculations or change the data type.

This approach preserves the records while avoiding unsupported assumptions.


In [ ]:
products_clean = products.copy()
products_clean["ProductColor"] = products_clean["ProductColor"].fillna("Unknown")


In [ ]:
audit_df(products_clean, "cleaned product data")

## Product Missing Value Handling

The products table contained missing values in the `ProductColor` column.

Since product color is a categorical attribute, missing values were filled with `Unknown`. This keeps the affected product records in the dataset while making the missing color information explicit for Power BI reporting.

No product rows were removed during this step.


In [ ]:
cleaned_tables = {
    "calendar_clean": calendar,
    "customers_clean": customers_clean,
    "product_categories_clean": product_categories,
    "products_clean": products_clean,
    "product_subcategories_clean": product_subcategories,
    "returns_clean": returns,
    "sales_clean": combined_sales,
    "territories_clean": territories
}

for table_name, df in cleaned_tables.items():
    output_path = clean_folder / f"{table_name}.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {table_name}: {len(df)} rows -> {output_path}")


## Additional Customer Data Validation

During review of the exported customer file, several invalid rows were found at the bottom of the customer table. These rows contained non-numeric `CustomerKey` values such as metadata/footer text rather than valid customer IDs.

The initial audit checked for missing values and duplicate rows, but did not check whether key fields followed the expected numeric format. Because `CustomerKey` is required for Power BI relationships, an additional validation check was added to confirm that all customer keys are numeric.

Invalid customer key rows were removed before re-exporting the cleaned customer table.


In [ ]:
customers_clean["CustomerKey_numeric"] = pd.to_numeric(
    customers_clean["CustomerKey"],
    errors="coerce"
)

invalid_customer_key_rows = customers_clean[customers_clean["CustomerKey_numeric"].isna()]
invalid_customer_key_rows

In [ ]:
customers_clean = customers_clean[customers_clean["CustomerKey_numeric"].notna()].copy()

customers_clean["CustomerKey"] = customers_clean["CustomerKey_numeric"].astype(int)

customers_clean = customers_clean.drop(columns=["CustomerKey_numeric"]).reset_index(drop=True)

In [ ]:
audit_df(customers_clean, "cleaned customer data")

In [ ]:
encoding_pattern = "Ã|Â|�|â"

customers_clean["possible_encoding_issue"] = (
    customers_clean["FirstName"].astype(str).str.contains(encoding_pattern, na=False, regex=True) |
    customers_clean["LastName"].astype(str).str.contains(encoding_pattern, na=False, regex=True) |
    customers_clean["EmailAddress"].astype(str).str.contains(encoding_pattern, na=False, regex=True)
)


In [ ]:
customers_clean["possible_encoding_issue"].value_counts()


In [ ]:
customers_clean[
    customers_clean["LastName"].astype(str).str.contains("NAVARRO", na=False)
][["CustomerKey", "FirstName", "LastName", "EmailAddress"]]


In [ ]:
customers_clean.to_csv(clean_folder / "customers_clean.csv", encoding="utf-8-sig")

The customer data was verified in pandas to confirm that accented names were stored correctly. Cleaned CSV files were exported using `utf-8-sig` encoding to improve compatibility with Excel and Power BI when displaying special characters.

Customer name fields contained some possible character encoding inconsistencies. Since names are not used as analytical fields in this Power BI dashboard, they were not mass-corrected to avoid accidentally altering valid names. The analysis focuses on validated key fields and business attributes such as customer demographics, products, territories, sales, and returns.


cleaned data ready to be analysed in PowerBI